# BBB v6 — Rigorous Nested Grouped Validation

**Author:** Moamen Elsayed  
**Programme:** M.Sc. Digital Chemistry, University of Gdańsk  
**Endpoint:** Quantitative whole-brain biodistribution (%ID/g)  
**Independent validation unit:** `Study_Group`

This notebook implements fully nested model/feature/transform/hyperparameter selection. Run the cells in order and upload `Inorganic_Nanoparticle_Brain_Uptake_Enriched_Primary_v5.xlsx` when prompted. The full run can take several minutes.


In [ ]:
"""
BBB v6: rigorous study-aware machine-learning pipeline
======================================================

Scientific scope
----------------
Quantitative whole-brain biodistribution (%ID/g) of inorganic nanoparticles.
This endpoint is not equivalent to demonstrated blood-brain-barrier crossing or
parenchymal localization.

Primary validity design
-----------------------
* Study_Group is the independent resampling unit.
* The outer GroupKFold loop is used only for final performance assessment.
* Feature-set, algorithm, target-transform, and hyperparameter selection occur
  exclusively inside each outer training partition.
* The primary reported performance is for that complete selection procedure,
  not for a model chosen after inspecting outer-fold performance.
* Model-family comparisons use identical outer splits, inner tuning, paired
  study-level errors, cluster bootstrap intervals, and paired sign-flip tests.
* Q2 is omitted because, under the former definition, it was mathematically
  identical to pooled out-of-fold R2.
* No global warning suppression is used.
* SHAP is omitted. Interpretability uses held-out permutation importance only.

Zero-target provenance
----------------------
Seven zero values come from one study (DOI 10.18869/acadpub.ijrr.18.3.539),
whose Table 1 reports brain uptake as 0.0 +/- 0.0 %ID/g. The paper does not
state an organ-specific LOD/LOQ, so these are retained as reported rounded
zeros, not interpreted as proven biological absence. A whole-study exclusion
sensitivity analysis is produced.
"""

from __future__ import annotations

import importlib.metadata
import importlib.util
import itertools
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable




In [ ]:
# ---------------------------------------------------------------------------
# 0. Reproducible dependency setup
# ---------------------------------------------------------------------------
REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "joblib": "joblib",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import rankdata, spearmanr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR


RANDOM_STATE = 42
OUTER_SPLITS = 5
INNER_SPLITS = 3
BOOTSTRAP_REPLICATES = 5000
PERMUTATION_REPLICATES = 20000
IMPORTANCE_REPEATS = 10
OUTPUT_DIR = Path("BBB_v6_Rigorous_results")
OUTPUT_DIR.mkdir(exist_ok=True)

EXPECTED_FILE = "Inorganic_Nanoparticle_Brain_Uptake_Enriched_Primary_v5.xlsx"
SHEET_NAME = "Enriched_Master_v4"
TARGET = "Brain_uptake_percentID_g"
GROUP_COL = "Study_Group"
ZERO_STUDY = "URL:http://ijrr.com/article-1-3074-en.pdf"
ZERO_SOURCE_DOI = "10.18869/acadpub.ijrr.18.3.539"


def choose_input_file() -> str:
    if Path(EXPECTED_FILE).exists():
        return EXPECTED_FILE
    candidates = sorted(Path(".").glob("*Enriched_Primary_v5*.xlsx"))
    if candidates:
        return str(candidates[0])
    try:
        from google.colab import files

        print("Upload Inorganic_Nanoparticle_Brain_Uptake_Enriched_Primary_v5.xlsx")
        uploaded = files.upload()
        choices = [name for name in uploaded if name.lower().endswith(".xlsx")]
        if not choices:
            raise RuntimeError("No .xlsx workbook was uploaded.")
        preferred = [name for name in choices if "enriched_primary_v5" in name.lower()]
        return preferred[0] if preferred else choices[0]
    except ImportError as exc:
        raise FileNotFoundError(
            f"Place {EXPECTED_FILE} in the current working directory."
        ) from exc




In [ ]:
# ---------------------------------------------------------------------------
# 1. Load, audit, and prepare data
# ---------------------------------------------------------------------------
INPUT_FILE = choose_input_file()
raw_df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

required_columns = [
    "Observation_ID",
    GROUP_COL,
    "Formulation_Group",
    "Material",
    "Time_h",
    TARGET,
    "QC_Status",
]
missing_required = [c for c in required_columns if c not in raw_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df = raw_df[
    raw_df["QC_Status"].astype(str).str.startswith("Accepted", na=False)
].copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df[df[TARGET].notna() & (df[TARGET] >= 0)].copy()
df = df[df[GROUP_COL].notna()].copy()
df = df[df[GROUP_COL].astype(str).str.strip().ne("")].reset_index(drop=True)
df[GROUP_COL] = df[GROUP_COL].astype(str)

numeric_candidates = [
    "Kumar_size_nm",
    "Kumar_PEG_MW_Da_if_numeric",
    "Primary_core_size_nm",
    "Hydrodynamic_size_nm",
    "PDI",
    "Length_nm",
    "Width_nm",
    "Aspect_ratio",
    "Zeta_potential_mV",
    "PEG_MW_Da_verified",
    "Time_h",
    "Age_weeks",
    "Weight_g",
    "Dose_mg_kg",
    "Number_of_doses",
]
for column in numeric_candidates:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].replace(
        {"": np.nan, "NA": np.nan, "N/A": np.nan, "nan": np.nan}
    )

df["Log1p_Time_h"] = np.log1p(df["Time_h"])
for source, derived in [
    ("Kumar_size_nm", "Log1p_Kumar_size_nm"),
    ("Primary_core_size_nm", "Log1p_Primary_core_size_nm"),
    ("Hydrodynamic_size_nm", "Log1p_Hydrodynamic_size_nm"),
]:
    if source in df.columns:
        values = pd.to_numeric(df[source], errors="coerce")
        df[derived] = np.where(values >= 0, np.log1p(values), np.nan)

if df[GROUP_COL].nunique() < OUTER_SPLITS:
    raise ValueError(
        f"Need at least {OUTER_SPLITS} independent Study_Group values; "
        f"found {df[GROUP_COL].nunique()}."
    )




In [ ]:
# ---------------------------------------------------------------------------
# 2. Fixed, scientifically defined feature sets
# ---------------------------------------------------------------------------
# No feature is selected from its association with the target. Missingness is
# handled inside each training fold. Identifiers and formulation names are never
# predictors.
FEATURE_SET_A = [
    "Material",
    "Kumar_size_nm",
    "Kumar_shape_standardized",
    "Kumar_charge_standardized",
]

FEATURE_SET_B = FEATURE_SET_A + [
    "Kumar_surface_modifier_standardized",
    "Kumar_PEG_status",
    "Kumar_PEG_MW_Da_if_numeric",
    "Primary_core_size_nm",
    "Hydrodynamic_size_nm",
    "PDI",
    "Shape_detail",
    "Coating_class",
    "Coating_material",
    "Zeta_potential_mV",
    "PEG_MW_Da_verified",
    "Targeting_ligand",
    "Ligand_class",
]

# Feature Set C deliberately combines nanoparticle descriptors with exposure,
# biological-context, and measurement variables. It must not be interpreted as
# merely "more nanoparticle properties."
FEATURE_SET_C = FEATURE_SET_B + [
    "Time_h",
    "Log1p_Time_h",
    "Strain_standardized",
    "Disease_status",
    "Disease_model",
    "BBB_intervention",
    "Sex",
    "Age_weeks",
    "Weight_g",
    "Dose_mg_kg",
    "Dosing_frequency",
    "Number_of_doses",
    "Label_or_isotope",
    "Analytical_method_primary_paper",
    "Perfusion_performed",
    "Brain_processing_method",
]

# Material sensitivity: identical to C but without Material, because every study
# in this dataset contains only one material and Material can proxy study origin.
FEATURE_SET_D = [c for c in FEATURE_SET_C if c != "Material"]

FEATURE_SETS = {
    "A_basic_nanoparticle": FEATURE_SET_A,
    "B_enriched_nanoparticle": FEATURE_SET_B,
    "C_nanoparticle_plus_experimental_context": FEATURE_SET_C,
    "D_full_context_without_material": FEATURE_SET_D,
}
FEATURE_SETS = {
    name: list(dict.fromkeys(c for c in columns if c in df.columns))
    for name, columns in FEATURE_SETS.items()
}

for name, columns in FEATURE_SETS.items():
    if not columns:
        raise ValueError(f"Feature set {name} contains no available columns.")




In [ ]:
# ---------------------------------------------------------------------------
# 3. Association and provenance audits
# ---------------------------------------------------------------------------
material_study_table = pd.crosstab(df[GROUP_COL], df["Material"])
materials_per_study = material_study_table.gt(0).sum(axis=1)
study_material_purity = float((materials_per_study == 1).mean())

zero_rows = df[df[TARGET].eq(0)].copy()
zero_audit = {
    "zero_rows": int(len(zero_rows)),
    "zero_study_groups": sorted(zero_rows[GROUP_COL].unique().tolist()),
    "published_source_doi": ZERO_SOURCE_DOI,
    "published_table_statement": "Brain values reported as 0.0 +/- 0.0 %ID/g.",
    "lod_loq_status": "Organ-specific LOD/LOQ was not reported in the paper.",
    "primary_treatment": "Retained as reported rounded zeros; not treated as missing.",
    "sensitivity_treatment": "Exclude the entire zero-containing Study_Group.",
}




In [ ]:
# ---------------------------------------------------------------------------
# 4. Preprocessing and candidate definitions
# ---------------------------------------------------------------------------
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_columns = [
        c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])
    ]
    categorical_columns = [c for c in X.columns if c not in numeric_columns]

    numeric_pipeline = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


MODEL_DEFINITIONS: dict[str, tuple[Any, dict[str, list[Any]]]] = {
    "Ridge": (
        Ridge(),
        {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    ),
    "ElasticNet": (
        ElasticNet(max_iter=30000, random_state=RANDOM_STATE),
        {
            "alpha": [0.001, 0.01, 0.1],
            "l1_ratio": [0.2, 0.5, 0.8],
        },
    ),
    "SVR_RBF": (
        SVR(kernel="rbf"),
        {
            "C": [0.5, 2.0, 10.0],
            "epsilon": [0.05, 0.2],
            "gamma": ["scale"],
        },
    ),
    "ExtraTrees": (
        ExtraTreesRegressor(
            n_estimators=250,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        {
            "max_depth": [3, None],
            "min_samples_leaf": [2, 5],
            "max_features": ["sqrt", 0.7],
        },
    ),
}
TARGET_TRANSFORMS = ["raw", "log1p"]


@dataclass(frozen=True)
class Candidate:
    family: str
    feature_set: str
    target_transform: str
    parameters_json: str

    @property
    def parameters(self) -> dict[str, Any]:
        return json.loads(self.parameters_json)


def all_candidates(family: str | None = None) -> list[Candidate]:
    families = [family] if family else list(MODEL_DEFINITIONS)
    candidates: list[Candidate] = []
    for model_family in families:
        _, grid = MODEL_DEFINITIONS[model_family]
        for feature_name, transform, params in itertools.product(
            FEATURE_SETS,
            TARGET_TRANSFORMS,
            ParameterGrid(grid),
        ):
            candidates.append(
                Candidate(
                    family=model_family,
                    feature_set=feature_name,
                    target_transform=transform,
                    parameters_json=json.dumps(params, sort_keys=True),
                )
            )
    return candidates


def build_estimator(X: pd.DataFrame, candidate: Candidate) -> Any:
    base_model, _ = MODEL_DEFINITIONS[candidate.family]
    model = clone(base_model).set_params(**candidate.parameters)
    pipeline = Pipeline(
        [
            ("preprocess", build_preprocessor(X)),
            ("model", model),
        ]
    )
    if candidate.target_transform == "log1p":
        return TransformedTargetRegressor(
            regressor=pipeline,
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False,
        )
    return pipeline


def nonnegative_predictions(estimator: Any, X: pd.DataFrame) -> np.ndarray:
    return np.maximum(np.asarray(estimator.predict(X), dtype=float), 0.0)




In [ ]:
# ---------------------------------------------------------------------------
# 5. Group-aware metrics and uncertainty
# ---------------------------------------------------------------------------
def rmse(y_true: Iterable[float], y_pred: Iterable[float]) -> float:
    return float(math.sqrt(mean_squared_error(y_true, y_pred)))


def group_error_table(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
) -> pd.DataFrame:
    temp = pd.DataFrame(
        {"Observed": y_true, "Predicted": y_pred, GROUP_COL: groups}
    )
    rows = []
    for group, part in temp.groupby(GROUP_COL, sort=True):
        rows.append(
            {
                GROUP_COL: group,
                "Rows": len(part),
                "Group_MAE": mean_absolute_error(part["Observed"], part["Predicted"]),
                "Group_RMSE": rmse(part["Observed"], part["Predicted"]),
                "Observed_mean": part["Observed"].mean(),
                "Predicted_mean": part["Predicted"].mean(),
            }
        )
    return pd.DataFrame(rows)


def point_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
) -> dict[str, float]:
    grouped = group_error_table(y_true, y_pred, groups)
    observation_rho = safe_spearman_statistic(y_true, y_pred)
    group_rho = safe_spearman_statistic(
        grouped["Observed_mean"].to_numpy(),
        grouped["Predicted_mean"].to_numpy(),
    )
    return {
        "MAE_observation_weighted": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_observation_weighted": rmse(y_true, y_pred),
        "R2_pooled_OOF": float(r2_score(y_true, y_pred)),
        "MAE_group_macro": float(grouped["Group_MAE"].mean()),
        "RMSE_group_macro": float(grouped["Group_RMSE"].mean()),
        "Spearman_observation_descriptive": float(observation_rho),
        "Spearman_group_means": float(group_rho),
    }


def safe_spearman_statistic(x: np.ndarray, y: np.ndarray) -> float:
    """Return NaN explicitly for constant inputs; do not suppress warnings."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(spearmanr(x, y).statistic)


def cluster_bootstrap_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
    n_boot: int = BOOTSTRAP_REPLICATES,
    seed: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    group_indices = [np.flatnonzero(groups == g) for g in unique_groups]
    group_mae = np.asarray(
        [mean_absolute_error(y_true[idx], y_pred[idx]) for idx in group_indices]
    )
    group_rmse = np.asarray([rmse(y_true[idx], y_pred[idx]) for idx in group_indices])
    group_observed_mean = np.asarray([np.mean(y_true[idx]) for idx in group_indices])
    group_predicted_mean = np.asarray([np.mean(y_pred[idx]) for idx in group_indices])
    records = []
    for _ in range(n_boot):
        sampled = rng.integers(0, len(unique_groups), size=len(unique_groups))
        row_indices = np.concatenate([group_indices[index] for index in sampled])
        yy = y_true[row_indices]
        pp = y_pred[row_indices]
        residual = yy - pp
        tss = np.sum((yy - np.mean(yy)) ** 2)
        r2 = np.nan if tss == 0 else 1.0 - np.sum(residual**2) / tss
        records.append(
            {
                "MAE_observation_weighted": float(np.mean(np.abs(residual))),
                "RMSE_observation_weighted": float(np.sqrt(np.mean(residual**2))),
                "R2_pooled_OOF": float(r2),
                "MAE_group_macro": float(np.mean(group_mae[sampled])),
                "RMSE_group_macro": float(np.mean(group_rmse[sampled])),
                "Spearman_observation_descriptive": safe_spearman_statistic(yy, pp),
                "Spearman_group_means": safe_spearman_statistic(
                    group_observed_mean[sampled], group_predicted_mean[sampled]
                ),
            }
        )
    return pd.DataFrame(records)


def summarize_with_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
    procedure: str,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    point = point_metrics(y_true, y_pred, groups)
    bootstrap = cluster_bootstrap_metrics(y_true, y_pred, groups, seed=seed)
    rows = []
    for metric, estimate in point.items():
        values = bootstrap[metric].dropna()
        rows.append(
            {
                "Procedure": procedure,
                "Metric": metric,
                "Estimate": estimate,
                "CI95_lower": float(values.quantile(0.025)),
                "CI95_upper": float(values.quantile(0.975)),
                "Bootstrap_unit": "Study_Group",
                "Bootstrap_replicates_used": len(values),
            }
        )
    return pd.DataFrame(rows), bootstrap.assign(Procedure=procedure)


def group_spearman_permutation_test(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
    n_permutations: int = PERMUTATION_REPLICATES,
    seed: int = RANDOM_STATE,
) -> tuple[float, float]:
    grouped = group_error_table(y_true, y_pred, groups)
    actual = grouped["Observed_mean"].to_numpy(dtype=float)
    predicted = grouped["Predicted_mean"].to_numpy(dtype=float)
    observed = safe_spearman_statistic(actual, predicted)
    if not np.isfinite(observed):
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    actual_rank = rankdata(actual)
    predicted_rank = rankdata(predicted)
    actual_centered = actual_rank - actual_rank.mean()
    denominator = np.sqrt(
        np.sum(actual_centered**2)
        * np.sum((predicted_rank - predicted_rank.mean()) ** 2)
    )
    extreme = 0
    for _ in range(n_permutations):
        permuted_rank = rng.permutation(predicted_rank)
        statistic = float(
            np.dot(actual_centered, permuted_rank - permuted_rank.mean())
            / denominator
        )
        extreme += int(abs(statistic) >= abs(observed))
    p_value = (extreme + 1) / (n_permutations + 1)
    return observed, float(p_value)




In [ ]:
# ---------------------------------------------------------------------------
# 6. Inner selection and outer assessment
# ---------------------------------------------------------------------------
def inner_score_candidate(train_data: pd.DataFrame, candidate: Candidate) -> float:
    columns = FEATURE_SETS[candidate.feature_set]
    X = train_data[columns]
    y = train_data[TARGET].astype(float)
    groups = train_data[GROUP_COL].astype(str)
    n_splits = min(INNER_SPLITS, groups.nunique())
    if n_splits < 2:
        return np.inf

    validation_records = []
    splitter = GroupKFold(n_splits=n_splits)
    for train_idx, validation_idx in splitter.split(X, y, groups):
        X_train = X.iloc[train_idx]
        X_validation = X.iloc[validation_idx]
        y_train = y.iloc[train_idx]
        y_validation = y.iloc[validation_idx]
        validation_groups = groups.iloc[validation_idx]

        estimator = build_estimator(X_train, candidate)
        estimator.fit(X_train, y_train)
        prediction = nonnegative_predictions(estimator, X_validation)
        # Selection uses only group-macro MAE. Avoid calculating unused squared
        # errors here: pathological losing candidates may extrapolate strongly,
        # and their RMSE is irrelevant to the prespecified selection criterion.
        validation_frame = pd.DataFrame(
            {
                GROUP_COL: validation_groups.to_numpy(),
                "Absolute_error": np.abs(y_validation.to_numpy() - prediction),
            }
        )
        fold_table = (
            validation_frame.groupby(GROUP_COL, as_index=False)["Absolute_error"]
            .mean()
            .rename(columns={"Absolute_error": "Group_MAE"})
        )
        validation_records.append(fold_table)

    all_group_errors = pd.concat(validation_records, ignore_index=True)
    return float(all_group_errors["Group_MAE"].mean())


def select_candidate(
    train_data: pd.DataFrame,
    candidates: list[Candidate],
) -> tuple[Candidate, pd.DataFrame]:
    rows = []
    for candidate in candidates:
        started = time.time()
        try:
            score = inner_score_candidate(train_data, candidate)
            failure = ""
        except Exception as exc:
            score = np.inf
            failure = repr(exc)
        rows.append(
            {
                **asdict(candidate),
                "Inner_group_macro_MAE": score,
                "Failure": failure,
                "Elapsed_seconds": time.time() - started,
            }
        )
    table = pd.DataFrame(rows).sort_values(
        ["Inner_group_macro_MAE", "family", "feature_set", "target_transform"],
        kind="stable",
    )
    valid = table[np.isfinite(table["Inner_group_macro_MAE"])]
    if valid.empty:
        raise RuntimeError("Every inner-CV candidate failed.")
    winner = valid.iloc[0]
    selected = Candidate(
        family=winner["family"],
        feature_set=winner["feature_set"],
        target_transform=winner["target_transform"],
        parameters_json=winner["parameters_json"],
    )
    return selected, table.reset_index(drop=True)


def fit_predict_candidate(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    candidate: Candidate,
) -> tuple[Any, np.ndarray]:
    columns = FEATURE_SETS[candidate.feature_set]
    estimator = build_estimator(train_data[columns], candidate)
    estimator.fit(train_data[columns], train_data[TARGET].astype(float))
    prediction = nonnegative_predictions(estimator, test_data[columns])
    return estimator, prediction


def held_out_permutation_importance(
    estimator: Any,
    test_data: pd.DataFrame,
    candidate: Candidate,
    baseline_prediction: np.ndarray,
    fold: int,
) -> pd.DataFrame:
    columns = FEATURE_SETS[candidate.feature_set]
    X_test = test_data[columns].copy()
    y_test = test_data[TARGET].to_numpy(dtype=float)
    g_test = test_data[GROUP_COL].astype(str).to_numpy()
    baseline = point_metrics(y_test, baseline_prediction, g_test)["MAE_group_macro"]
    rng = np.random.default_rng(RANDOM_STATE + fold)
    rows = []
    for column in columns:
        increases = []
        for _ in range(IMPORTANCE_REPEATS):
            permuted = X_test.copy()
            permuted[column] = rng.permutation(permuted[column].to_numpy())
            prediction = nonnegative_predictions(estimator, permuted)
            permuted_mae = point_metrics(y_test, prediction, g_test)[
                "MAE_group_macro"
            ]
            increases.append(permuted_mae - baseline)
        rows.append(
            {
                "Outer_fold": fold,
                "Selected_family": candidate.family,
                "Selected_feature_set": candidate.feature_set,
                "Feature": column,
                "Importance_MAE_increase_mean": float(np.mean(increases)),
                "Importance_MAE_increase_SD": float(np.std(increases, ddof=1)),
                "Repeats": IMPORTANCE_REPEATS,
                "Evaluation_data": "Held-out outer-fold studies",
            }
        )
    return pd.DataFrame(rows)


def run_nested_assessment(data: pd.DataFrame, analysis_name: str) -> dict[str, Any]:
    outer_splitter = GroupKFold(n_splits=min(OUTER_SPLITS, data[GROUP_COL].nunique()))
    X_stub = data[[GROUP_COL]]
    y = data[TARGET].astype(float)
    groups = data[GROUP_COL].astype(str)

    procedures = ["Nested_selection_procedure", *MODEL_DEFINITIONS, "DummyMean"]
    oof = {name: np.full(len(data), np.nan, dtype=float) for name in procedures}
    outer_rows = []
    selection_rows = []
    inner_tables = []
    importance_tables = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_splitter.split(X_stub, y, groups), start=1
    ):
        train_data = data.iloc[train_idx].copy()
        test_data = data.iloc[test_idx].copy()
        held_out = sorted(test_data[GROUP_COL].unique())
        print(
            f"[{analysis_name}] outer fold {fold}/{OUTER_SPLITS}: "
            f"{len(held_out)} held-out studies"
        )

        # Primary procedure: all choices are made in the outer training data.
        selected, inner_table = select_candidate(train_data, all_candidates())
        inner_table["Analysis"] = analysis_name
        inner_table["Outer_fold"] = fold
        inner_table["Selection_scope"] = "All families"
        inner_tables.append(inner_table)

        estimator, prediction = fit_predict_candidate(train_data, test_data, selected)
        oof["Nested_selection_procedure"][test_idx] = prediction
        importance_tables.append(
            held_out_permutation_importance(
                estimator, test_data, selected, prediction, fold
            ).assign(Analysis=analysis_name)
        )
        selection_rows.append(
            {
                "Analysis": analysis_name,
                "Outer_fold": fold,
                "Procedure": "Nested_selection_procedure",
                **asdict(selected),
                "Held_out_studies": " | ".join(held_out),
            }
        )

        # Family-specific nested procedures on the identical outer split.
        for family in MODEL_DEFINITIONS:
            family_valid = inner_table[
                inner_table["family"].eq(family)
                & np.isfinite(inner_table["Inner_group_macro_MAE"])
            ]
            if family_valid.empty:
                raise RuntimeError(f"Every inner-CV candidate failed for {family}.")
            family_winner = family_valid.iloc[0]
            family_selected = Candidate(
                family=family_winner["family"],
                feature_set=family_winner["feature_set"],
                target_transform=family_winner["target_transform"],
                parameters_json=family_winner["parameters_json"],
            )
            _, family_prediction = fit_predict_candidate(
                train_data, test_data, family_selected
            )
            oof[family][test_idx] = family_prediction
            selection_rows.append(
                {
                    "Analysis": analysis_name,
                    "Outer_fold": fold,
                    "Procedure": family,
                    **asdict(family_selected),
                    "Held_out_studies": " | ".join(held_out),
                }
            )

        dummy = DummyRegressor(strategy="mean")
        dummy.fit(np.zeros((len(train_data), 1)), train_data[TARGET])
        oof["DummyMean"][test_idx] = dummy.predict(
            np.zeros((len(test_data), 1))
        )

        for procedure, predictions in oof.items():
            fold_prediction = predictions[test_idx]
            if np.isnan(fold_prediction).any():
                continue
            fold_metrics = point_metrics(
                test_data[TARGET].to_numpy(dtype=float),
                fold_prediction,
                test_data[GROUP_COL].astype(str).to_numpy(),
            )
            outer_rows.append(
                {
                    "Analysis": analysis_name,
                    "Outer_fold": fold,
                    "Procedure": procedure,
                    "Train_rows": len(train_data),
                    "Test_rows": len(test_data),
                    "Train_studies": train_data[GROUP_COL].nunique(),
                    "Test_studies": test_data[GROUP_COL].nunique(),
                    "Held_out_studies": " | ".join(held_out),
                    **fold_metrics,
                }
            )

    prediction_tables = []
    metric_tables = []
    bootstrap_tables = []
    group_error_tables = []
    spearman_rows = []
    for index, (procedure, predictions) in enumerate(oof.items()):
        if np.isnan(predictions).any():
            raise RuntimeError(f"Missing outer predictions for {procedure}.")
        pred_table = data[
            ["Observation_ID", GROUP_COL, "Formulation_Group", "Material", TARGET]
        ].copy()
        pred_table = pred_table.rename(columns={TARGET: "Observed"})
        pred_table["Predicted"] = predictions
        pred_table["Residual"] = pred_table["Observed"] - pred_table["Predicted"]
        pred_table["Procedure"] = procedure
        pred_table["Analysis"] = analysis_name
        prediction_tables.append(pred_table)

        metrics, bootstrap = summarize_with_ci(
            data[TARGET].to_numpy(dtype=float),
            predictions,
            data[GROUP_COL].astype(str).to_numpy(),
            procedure,
            seed=RANDOM_STATE + 100 * index,
        )
        metrics["Analysis"] = analysis_name
        bootstrap["Analysis"] = analysis_name
        metric_tables.append(metrics)
        bootstrap_tables.append(bootstrap)

        group_errors = group_error_table(
            data[TARGET].to_numpy(dtype=float),
            predictions,
            data[GROUP_COL].astype(str).to_numpy(),
        )
        group_errors["Procedure"] = procedure
        group_errors["Analysis"] = analysis_name
        group_error_tables.append(group_errors)

        rho, permutation_p = group_spearman_permutation_test(
            data[TARGET].to_numpy(dtype=float),
            predictions,
            data[GROUP_COL].astype(str).to_numpy(),
            seed=RANDOM_STATE + 1000 * index,
        )
        spearman_rows.append(
            {
                "Analysis": analysis_name,
                "Procedure": procedure,
                "Spearman_group_means": rho,
                "Permutation_p_two_sided": permutation_p,
                "Independent_units": data[GROUP_COL].nunique(),
                "Permutation_unit": "Study_Group mean",
                "Permutation_replicates": PERMUTATION_REPLICATES,
            }
        )

    return {
        "predictions": pd.concat(prediction_tables, ignore_index=True),
        "metrics": pd.concat(metric_tables, ignore_index=True),
        "bootstraps": pd.concat(bootstrap_tables, ignore_index=True),
        "group_errors": pd.concat(group_error_tables, ignore_index=True),
        "spearman": pd.DataFrame(spearman_rows),
        "outer_folds": pd.DataFrame(outer_rows),
        "selections": pd.DataFrame(selection_rows),
        "inner_scores": pd.concat(inner_tables, ignore_index=True),
        "importance": pd.concat(importance_tables, ignore_index=True),
    }




In [ ]:
# ---------------------------------------------------------------------------
# 7. Paired model-family comparisons at Study_Group level
# ---------------------------------------------------------------------------
def holm_adjust(p_values: list[float]) -> list[float]:
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running = 0.0
    m = len(p_values)
    for rank, index in enumerate(order):
        value = min(1.0, (m - rank) * p_values[index])
        running = max(running, value)
        adjusted[index] = running
    return adjusted.tolist()


def paired_sign_flip_test(
    differences: np.ndarray,
    seed: int,
    n_permutations: int = PERMUTATION_REPLICATES,
) -> float:
    differences = differences[np.isfinite(differences)]
    observed = abs(float(np.mean(differences)))
    rng = np.random.default_rng(seed)
    extreme = 0
    for _ in range(n_permutations):
        signs = rng.choice([-1.0, 1.0], size=len(differences))
        permuted = abs(float(np.mean(differences * signs)))
        extreme += int(permuted >= observed)
    return float((extreme + 1) / (n_permutations + 1))


def paired_comparisons(group_errors: pd.DataFrame) -> pd.DataFrame:
    wide = group_errors.pivot(
        index=GROUP_COL, columns="Procedure", values="Group_MAE"
    )
    procedures = list(wide.columns)
    rng = np.random.default_rng(RANDOM_STATE + 9000)
    rows = []
    for comparison_index, (left, right) in enumerate(
        itertools.combinations(procedures, 2)
    ):
        pair = wide[[left, right]].dropna()
        differences = (pair[left] - pair[right]).to_numpy(dtype=float)
        boot_means = []
        for _ in range(BOOTSTRAP_REPLICATES):
            sample = rng.choice(differences, size=len(differences), replace=True)
            boot_means.append(float(np.mean(sample)))
        rows.append(
            {
                "Procedure_A": left,
                "Procedure_B": right,
                "Mean_group_MAE_difference_A_minus_B": float(np.mean(differences)),
                "CI95_lower": float(np.quantile(boot_means, 0.025)),
                "CI95_upper": float(np.quantile(boot_means, 0.975)),
                "Paired_sign_flip_p": paired_sign_flip_test(
                    differences, seed=RANDOM_STATE + comparison_index
                ),
                "N_study_groups": len(differences),
                "Interpretation": (
                    "Negative favors Procedure_A; positive favors Procedure_B."
                ),
            }
        )
    table = pd.DataFrame(rows)
    table["Holm_adjusted_p"] = holm_adjust(table["Paired_sign_flip_p"].tolist())
    return table




In [ ]:
# ---------------------------------------------------------------------------
# 8. Run primary and zero-study sensitivity analyses
# ---------------------------------------------------------------------------
primary = run_nested_assessment(df, "Primary_reported_zeros_retained")

zero_sensitivity_data = df[df[GROUP_COL] != ZERO_STUDY].reset_index(drop=True)
zero_sensitivity = run_nested_assessment(
    zero_sensitivity_data,
    "Sensitivity_zero_source_study_excluded",
)

comparison_primary = paired_comparisons(primary["group_errors"])
comparison_primary["Analysis"] = "Primary_reported_zeros_retained"
comparison_sensitivity = paired_comparisons(zero_sensitivity["group_errors"])
comparison_sensitivity["Analysis"] = "Sensitivity_zero_source_study_excluded"
comparisons = pd.concat(
    [comparison_primary, comparison_sensitivity], ignore_index=True
)




In [ ]:
# ---------------------------------------------------------------------------
# 9. Select and fit a future-use final model without reusing it for assessment
# ---------------------------------------------------------------------------
final_candidate, final_selection_table = select_candidate(df, all_candidates())
final_columns = FEATURE_SETS[final_candidate.feature_set]
final_model = build_estimator(df[final_columns], final_candidate)
final_model.fit(df[final_columns], df[TARGET].astype(float))
joblib.dump(
    {
        "estimator": final_model,
        "feature_columns": final_columns,
        "candidate": asdict(final_candidate),
        "training_scope": "All available data; not used to estimate performance.",
    },
    OUTPUT_DIR / "final_model_for_future_predictions.joblib",
)




In [ ]:
# ---------------------------------------------------------------------------
# 10. Save auditable outputs
# ---------------------------------------------------------------------------
def save_csv(frame: pd.DataFrame, filename: str) -> None:
    frame.to_csv(OUTPUT_DIR / filename, index=False)


save_csv(primary["metrics"], "01_primary_metrics_with_cluster_CI.csv")
save_csv(primary["predictions"], "02_primary_outer_predictions.csv")
save_csv(primary["group_errors"], "03_primary_study_level_errors.csv")
save_csv(primary["outer_folds"], "04_primary_outer_fold_metrics.csv")
save_csv(primary["selections"], "05_primary_inner_selected_configurations.csv")
save_csv(primary["inner_scores"], "06_primary_all_inner_scores.csv")
save_csv(primary["spearman"], "07_primary_group_spearman_permutation.csv")
save_csv(comparisons, "08_paired_procedure_comparisons.csv")
save_csv(primary["importance"], "09_held_out_permutation_importance.csv")
save_csv(zero_sensitivity["metrics"], "10_zero_study_sensitivity_metrics.csv")
save_csv(zero_sensitivity["predictions"], "11_zero_study_sensitivity_predictions.csv")
save_csv(zero_sensitivity["group_errors"], "12_zero_study_sensitivity_group_errors.csv")
save_csv(final_selection_table, "13_full_data_inner_selection_for_final_fit.csv")
save_csv(material_study_table.reset_index(), "14_material_by_study_contingency.csv")

feature_rows = []
for feature_set, columns in FEATURE_SETS.items():
    for column in columns:
        feature_rows.append(
            {
                "Feature_Set": feature_set,
                "Feature": column,
                "Role": (
                    "Nanoparticle descriptor"
                    if feature_set != "C_nanoparticle_plus_experimental_context"
                    else "Mixed nanoparticle and experimental-context set"
                ),
                "Nonmissing_fraction": float(df[column].notna().mean()),
            }
        )
save_csv(pd.DataFrame(feature_rows), "15_feature_sets_and_coverage.csv")

importance_summary = (
    primary["importance"]
    .groupby("Feature", as_index=False)
    .agg(
        Outer_folds_selected=("Outer_fold", "nunique"),
        Mean_MAE_increase=("Importance_MAE_increase_mean", "mean"),
        Median_MAE_increase=("Importance_MAE_increase_mean", "median"),
        SD_across_folds=("Importance_MAE_increase_mean", "std"),
    )
    .sort_values("Mean_MAE_increase", ascending=False)
)
save_csv(importance_summary, "16_held_out_importance_stability.csv")

environment = {
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "python": sys.version,
    "python_implementation": platform.python_implementation(),
    "platform": platform.platform(),
    "packages": {
        name: importlib.metadata.version(distribution)
        for name, distribution in REQUIRED_PACKAGES.items()
    },
    "random_state": RANDOM_STATE,
    "outer_splits": OUTER_SPLITS,
    "inner_splits": INNER_SPLITS,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "permutation_replicates": PERMUTATION_REPLICATES,
}
(OUTPUT_DIR / "environment_versions.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "zero_target_provenance.json").write_text(
    json.dumps(zero_audit, indent=2), encoding="utf-8"
)




In [ ]:
# ---------------------------------------------------------------------------
# 11. Diagnostic plots based only on outer held-out predictions
# ---------------------------------------------------------------------------
primary_selected = primary["predictions"][
    primary["predictions"]["Procedure"] == "Nested_selection_procedure"
].copy()

plt.figure(figsize=(6.5, 6.0))
plt.scatter(primary_selected["Observed"], primary_selected["Predicted"], alpha=0.75)
low = min(primary_selected["Observed"].min(), primary_selected["Predicted"].min())
high = max(primary_selected["Observed"].max(), primary_selected["Predicted"].max())
plt.plot([low, high], [low, high], "--", color="black", linewidth=1)
plt.xlabel("Observed brain biodistribution (%ID/g)")
plt.ylabel("Outer held-out prediction (%ID/g)")
plt.title("Nested selection procedure: held-out predictions")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "17_observed_vs_outer_prediction.png", dpi=220)
plt.close()

metric_plot = primary["metrics"]
metric_plot = metric_plot[metric_plot["Metric"] == "MAE_group_macro"].copy()
metric_plot = metric_plot.sort_values("Estimate")
errors = np.vstack(
    [
        metric_plot["Estimate"] - metric_plot["CI95_lower"],
        metric_plot["CI95_upper"] - metric_plot["Estimate"],
    ]
)
plt.figure(figsize=(9, 5.5))
plt.errorbar(
    metric_plot["Estimate"],
    range(len(metric_plot)),
    xerr=errors,
    fmt="o",
    capsize=4,
)
plt.yticks(range(len(metric_plot)), metric_plot["Procedure"])
plt.xlabel("Study-macro MAE (%ID/g), 95% cluster-bootstrap CI")
plt.title("Outer assessment of prespecified procedures")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "18_procedure_MAE_with_CI.png", dpi=220)
plt.close()




In [ ]:
# ---------------------------------------------------------------------------
# 12. Human-readable scientific report
# ---------------------------------------------------------------------------
primary_summary = primary["metrics"].pivot(
    index="Procedure", columns="Metric", values="Estimate"
)
selected_metrics = primary_summary.loc["Nested_selection_procedure"]
dummy_metrics = primary_summary.loc["DummyMean"]

def metric_ci(table: pd.DataFrame, procedure: str, metric: str) -> tuple[float, float]:
    row = table[
        table["Procedure"].eq(procedure) & table["Metric"].eq(metric)
    ].iloc[0]
    return float(row["CI95_lower"]), float(row["CI95_upper"])


selected_mae_ci = metric_ci(
    primary["metrics"], "Nested_selection_procedure", "MAE_group_macro"
)
selected_rmse_ci = metric_ci(
    primary["metrics"], "Nested_selection_procedure", "RMSE_observation_weighted"
)
selected_r2_ci = metric_ci(
    primary["metrics"], "Nested_selection_procedure", "R2_pooled_OOF"
)
selected_spearman_row = primary["spearman"][
    primary["spearman"]["Procedure"].eq("Nested_selection_procedure")
].iloc[0]
sensitivity_selected = zero_sensitivity["metrics"].pivot(
    index="Procedure", columns="Metric", values="Estimate"
).loc["Nested_selection_procedure"]
primary_pairwise_min_holm = float(comparison_primary["Holm_adjusted_p"].min())
r2_value = float(selected_metrics["R2_pooled_OOF"])
variance_percent = 100.0 * max(r2_value, 0.0)

if r2_value <= 0:
    r2_interpretation = (
        "The pooled outer out-of-fold R2 is non-positive. The complete model-"
        "selection procedure does not outperform prediction by the training mean "
        "in explained-variance terms for unseen studies."
    )
elif r2_value < 0.10:
    r2_interpretation = (
        f"The pooled outer out-of-fold R2 is {r2_value:.4f}, corresponding to "
        f"only about {variance_percent:.2f}% of target variance explained. This "
        "is very weak predictive performance and is not practically persuasive "
        "evidence of generalization."
    )
else:
    r2_interpretation = (
        f"The pooled outer out-of-fold R2 is {r2_value:.4f}. Its practical value "
        "must be judged together with the uncertainty interval and dummy baseline."
    )

selection_counts = (
    primary["selections"]
    .query("Procedure == 'Nested_selection_procedure'")
    .groupby(["family", "feature_set", "target_transform"], dropna=False)
    .size()
    .sort_values(ascending=False)
)

report = f"""
BBB v6 RIGOROUS NESTED GROUPED VALIDATION
=========================================

DATA AND ENDPOINT
-----------------
Rows: {len(df)}
Independent Study_Group values: {df[GROUP_COL].nunique()}
Formulations: {df['Formulation_Group'].nunique()}
Materials: {', '.join(sorted(df['Material'].dropna().astype(str).unique()))}
Endpoint: quantitative whole-brain biodistribution (%ID/g), not demonstrated
BBB crossing or parenchymal localization.

PRIMARY VALIDATION ESTIMAND
---------------------------
Performance of the complete selection procedure on unseen Study_Group values.
Within every outer training partition, inner grouped CV selects the feature set,
model family, target transform, and hyperparameters. The outer test groups do not
participate in any selection decision.

PRIMARY NESTED-SELECTION RESULTS
--------------------------------
Study-macro MAE: {selected_metrics['MAE_group_macro']:.6f} %ID/g
95% group-bootstrap CI: [{selected_mae_ci[0]:.6f}, {selected_mae_ci[1]:.6f}]
Observation-weighted MAE: {selected_metrics['MAE_observation_weighted']:.6f} %ID/g
Observation-weighted RMSE: {selected_metrics['RMSE_observation_weighted']:.6f} %ID/g
95% group-bootstrap CI: [{selected_rmse_ci[0]:.6f}, {selected_rmse_ci[1]:.6f}]
Pooled outer OOF R2: {r2_value:.6f}
95% group-bootstrap CI: [{selected_r2_ci[0]:.6f}, {selected_r2_ci[1]:.6f}]
Group-mean Spearman rho: {selected_metrics['Spearman_group_means']:.6f}
Group-level permutation p-value: {selected_spearman_row['Permutation_p_two_sided']:.6f}

Dummy study-aware baseline study-macro MAE: {dummy_metrics['MAE_group_macro']:.6f} %ID/g

R2 INTERPRETATION
-----------------
{r2_interpretation}

Q2
---
Q2 is intentionally omitted. Under the definition used in earlier scripts, Q2
was exactly the pooled out-of-fold R2 and was not an independent metric.

UNCERTAINTY AND MODEL COMPARISONS
---------------------------------
All main metrics have 95% cluster-bootstrap intervals using Study_Group as the
resampling unit. Family procedures were evaluated on identical outer splits.
Paired differences in study-level MAE are accompanied by bootstrap intervals,
paired sign-flip permutation p-values, and Holm multiplicity correction. These
comparisons assess both statistical uncertainty and effect magnitude.
Minimum Holm-adjusted p-value across the primary pairwise comparisons:
{primary_pairwise_min_holm:.6f}. No algorithmic advantage should be claimed when
the adjusted comparison is not significant and its effect interval includes zero.

GROUP-AWARE CORRELATION
-----------------------
The inferential Spearman analysis uses one observed mean and one predicted mean
per Study_Group and a permutation test across study units. Observation-level
Spearman correlation is retained only as a descriptive statistic and has no
naive row-level p-value.

ZERO TARGETS
------------
Seven values from {ZERO_STUDY} are explicitly reported in Table 1 of DOI
{ZERO_SOURCE_DOI} as brain uptake 0.0 +/- 0.0 %ID/g. The publication does not
provide an organ-specific detection or quantification limit. They are therefore
retained as reported rounded zeros without claiming proven absence of uptake.
A sensitivity analysis excludes that entire study group.
Sensitivity nested-selection study-macro MAE:
{sensitivity_selected['MAE_group_macro']:.6f} %ID/g.
Sensitivity pooled outer OOF R2: {sensitivity_selected['R2_pooled_OOF']:.6f}.

MATERIAL AND STUDY_GROUP
------------------------
Proportion of studies containing exactly one material: {study_material_purity:.3f}.
Thus Study_Group perfectly determines Material within this dataset. Material is
a scientifically meaningful descriptor but also a study-level characteristic,
so its importance must not be interpreted as independent causal evidence. A
prespecified feature set excluding Material is included in inner selection.

FEATURE-SET INTERPRETATION
--------------------------
A: basic nanoparticle descriptors.
B: enriched nanoparticle descriptors.
C: nanoparticle descriptors PLUS experimental, biological, and measurement
   context. B-to-C differences are not solely effects of nanoparticle properties.
D: the full-context sensitivity set without Material.

INTERPRETABILITY
----------------
SHAP from an all-data fitted model is omitted. Reported permutation importance is
computed only on outer held-out studies. Stability is summarized across outer
folds; features absent from a fold's selected feature set receive no importance
claim for that fold.

FINAL FUTURE-USE MODEL
----------------------
After unbiased assessment was complete, inner grouped CV on all available data
selected:
{final_candidate}
This final all-data fit is stored only for future prediction. Its training fit is
not used as evidence of predictive performance.

OUTER-FOLD SELECTION FREQUENCIES
--------------------------------
{selection_counts.to_string()}

SOFTWARE
--------
Exact Python and package versions are recorded in environment_versions.json.
No global warnings are suppressed.
"""

(OUTPUT_DIR / "README_RESULTS.txt").write_text(report.strip() + "\n", encoding="utf-8")

methodology = {
    "input_file": str(INPUT_FILE),
    "sheet": SHEET_NAME,
    "target": TARGET,
    "group": GROUP_COL,
    "feature_sets": FEATURE_SETS,
    "model_families": list(MODEL_DEFINITIONS),
    "target_transforms": TARGET_TRANSFORMS,
    "primary_procedure": "Nested selection of feature set, family, transform, and hyperparameters",
    "performance_data": "Outer held-out Study_Group values only",
    "uncertainty": "Study_Group cluster bootstrap",
    "correlation_inference": "Spearman on Study_Group means with permutation p-value",
    "q2": "Omitted because it duplicated pooled OOF R2 under the previous definition",
    "shap": "Omitted; held-out permutation importance used instead",
    "zero_target_audit": zero_audit,
    "material_study_purity": study_material_purity,
}
(OUTPUT_DIR / "methodology.json").write_text(
    json.dumps(methodology, indent=2), encoding="utf-8"
)

print(report)

zip_path = shutil.make_archive(
    "BBB_v6_Rigorous_results", "zip", root_dir=OUTPUT_DIR
)
print("Results directory:", OUTPUT_DIR.resolve())
print("Results ZIP:", Path(zip_path).resolve())

try:
    from google.colab import files

    files.download(zip_path)
except ImportError:
    pass
